# GDP growth forecasting: models and validation

Author: Sami Haroon Khan (modelling and validation workstream)

Input: the four time-based split CSVs in `../merged_data/merged_data/`. Every row is one country and one input year t, and the target `next_year_gdp_growth` is GDP growth in year t+1.

Order of the notebook:
1. Load the data, add lag features, run leakage checks.
2. Add the region and income group for each country.
3. Build a naive baseline and 4 models, tune them on 2018-2019, then test on 2020 and 2021-2024.
4. Feature importance and the report figures.
5. Comparison with the IMF forecasts.

## 1. Load the four splits into one panel

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("../merged_data/merged_data")
OUT = Path("outputs")
(OUT / "figures").mkdir(parents=True, exist_ok=True)

files = {"train": "GDP_Training_2001_2017.csv", "val": "GDP_Validation_2018_2019.csv",
         "covid": "GDP_COVID_2020.csv", "test": "GDP_Testing_2021_2024.csv"}
panel = pd.concat([pd.read_csv(DATA / f).assign(split=s) for s, f in files.items()], ignore_index=True)
panel = panel.sort_values(["Country Code", "Year"]).reset_index(drop=True)
TARGET = "next_year_gdp_growth"
panel.groupby("split").size()

split
covid     217
test      868
train    3689
val       434
dtype: int64

## Leakage checks

- Every country must have all 24 input years (2000-2023), so that "the row above" really means "last year".
- The target year must always be the input year + 1.
- The four splits must not share any target year.

In [2]:
assert panel.groupby("Country Code")["Year"].apply(lambda y: list(y) == list(range(2000, 2024))).all()
assert (panel["target_year"] == panel["Year"] + 1).all()
yrs = panel.groupby("split")["target_year"].agg(["min", "max"])
assert yrs.loc["train", "max"] < yrs.loc["val", "min"] <= yrs.loc["val", "max"] < yrs.loc["covid", "min"] < yrs.loc["test", "min"]
yrs

,min,max
split,,
covid,2020,2020
test,2021,2024
train,2001,2017
val,2018,2019


## Lag features and filling gaps from the past

- Two extra features: last year's growth (`gdp_growth_lag1`), and the average growth over the last 3 years (`gdp_growth_avg3`). That brings the total to 10 features, within the guideline of about 10.
- A gap is filled with the same country's most recent earlier value. The rows are sorted by year, so this never looks into the future.
- The naive baseline guesses that next year's growth equals this year's growth.

In [3]:
g = panel.groupby("Country Code")["gdp_growth"]
panel["gdp_growth_lag1"] = g.shift(1)
panel["gdp_growth_avg3"] = g.transform(lambda s: s.rolling(3, min_periods=2).mean())

BASE = ["gdp_growth", "inflation", "unemployment", "trade", "investment",
        "population_growth", "government_debt", "current_account"]
FEATURES = BASE + ["gdp_growth_lag1", "gdp_growth_avg3"]

panel[FEATURES] = panel.groupby("Country Code")[FEATURES].ffill()
panel["naive"] = panel["gdp_growth"]
panel[FEATURES].isna().mean().mul(100).round(1).rename("% still missing")

gdp_growth            3.4
inflation            14.6
unemployment         13.8
trade                15.5
investment           18.8
population_growth     0.0
government_debt      14.2
current_account      13.6
gdp_growth_lag1       7.5
gdp_growth_avg3       7.9
Name: % still missing, dtype: float64

## 2. Region and income group

These come from the public World Bank country API. They are fetched once and saved to `outputs/country_groups.csv`, so the notebook runs offline after that.

Limitation: this is today's income classification, applied to every year.

In [4]:
gfile = OUT / "country_groups.csv"
if not gfile.exists():
    import requests
    js = requests.get("https://api.worldbank.org/v2/country?format=json&per_page=400", timeout=30).json()[1]
    pd.DataFrame([{"Country Code": c["id"], "region": c["region"]["value"],
                   "income_group": c["incomeLevel"]["value"]} for c in js]).to_csv(gfile, index=False)
groups = pd.read_csv(gfile)
missing = set(panel["Country Code"]) - set(groups["Country Code"])
assert not missing, missing
groups[groups["Country Code"].isin(panel["Country Code"])]["income_group"].value_counts()

income_group
High income            86
Upper middle income    59
Lower middle income    47
Low income             25
Name: count, dtype: int64